# 03 - Feature Engineering

Runs `backend/features/engineering.py` on the raw data and inspects the engineered features: velocity windows, amount deviation, device/IP novelty, and geo deviation.

In [1]:
import sys
sys.path.insert(0, '.')
import pandas as pd
from backend.features.engineering import load_raw, build_features
from pathlib import Path

txns, customers, devices, ips, merchants = load_raw(Path('data/processed'))
df, feature_cols = build_features(txns, customers, devices, ips, merchants)
print(f'{len(df):,} rows, {len(feature_cols)} features')
feature_cols

97,424 rows, 18 features


['amount',
 'amount_dev_ratio',
 'amount_zscore_merchant',
 'account_age_days',
 'transaction_frequency_per_week',
 'historical_chargebacks',
 'historical_refunds',
 'is_foreign_country',
 'device_is_new',
 'device_shared_flag',
 'ip_shared_flag',
 'device_customer_count',
 'ip_customer_count',
 'velocity_5min',
 'velocity_1hr',
 'seconds_since_prev_txn',
 'failed_txn_count_24h',
 'historical_fraud_rate']

## Feature summary statistics, split by fraud label

In [2]:
df.groupby('label')[feature_cols].mean().T.rename(columns={0: 'legit_mean', 1: 'fraud_mean'})

label,legit_mean,fraud_mean
amount,6.279480e+02,1594.436587
amount_dev_ratio,1.000527e+00,3.383510
amount_zscore_merchant,7.255869e-02,1.786678
account_age_days,1.000302e+03,1021.255404
transaction_frequency_per_week,3.005877e+00,3.005267
historical_chargebacks,2.295496e-01,2.636519
historical_refunds,3.835052e-01,0.562381
is_foreign_country,0.000000e+00,0.126090
device_is_new,8.631579e-02,0.106560
device_shared_flag,0.000000e+00,0.196815


## Velocity features clearly separate the velocity-attack scenario

In [3]:
scenario_labels = pd.read_csv('data/processed/transactions_scenario_labels.csv')
df2 = df.merge(scenario_labels, on='transaction_id', how='left')
df2.groupby('scenario')[['velocity_5min', 'velocity_1hr']].mean().sort_values('velocity_5min', ascending=False)

,velocity_5min,velocity_1hr
scenario,,
velocity_attack,5.029774,5.172998
card_testing,3.598988,4.579380
account_takeover,1.495146,1.648544
abuse_ring,1.004630,1.032407
legit,1.000098,1.001606
geo_anomaly,1.000000,1.586667


## Device/IP sharing flags clearly separate the abuse-ring scenario

In [4]:
df2.groupby('scenario')[['device_shared_flag', 'ip_shared_flag', 'device_customer_count', 'ip_customer_count']].mean()\
    .sort_values('ip_customer_count', ascending=False)

,device_shared_flag,ip_shared_flag,device_customer_count,ip_customer_count
scenario,,,,
abuse_ring,0.961111,0.990741,5.416667,6.402778
account_takeover,0.000000,0.000000,1.000000,1.000000
card_testing,0.000000,0.000000,1.000000,1.000000
geo_anomaly,0.000000,0.000000,1.000000,1.000000
legit,0.000000,0.000000,1.000000,1.000000
velocity_attack,0.000000,0.000000,1.000000,1.000000


## Save engineered feature sample for reference

In [5]:
df.head(1000).to_csv('data/processed/features_sample.csv', index=False)
print('Saved data/processed/features_sample.csv')

Saved data/processed/features_sample.csv
